In [12]:
# Data handling
import pandas as pd

# Text cleaning
import re

# Deep Learning
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM,Dense

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [13]:
df = pd.read_csv("chat_dataset.csv")

In [14]:
df.head()

,message,sentiment
0,I really enjoyed the movie,positive
1,The food was terrible,negative
2,I'm not sure how I feel about this,neutral
3,The service was excellent,positive
4,I had a bad experience,negative


In [15]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   message    584 non-null    str  
 1   sentiment  584 non-null    str  
dtypes: str(2)
memory usage: 9.3 KB
None


In [16]:
print(df.isnull().sum())

message      0
sentiment    0
dtype: int64


In [17]:
# Function to clean messages
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text

# Apply cleaning
df['clean_message'] = df['message'].apply(clean_text)

# Show cleaned messages
df[['message', 'clean_message']].head()

,message,clean_message
0,I really enjoyed the movie,i really enjoyed the movie
1,The food was terrible,the food was terrible
2,I'm not sure how I feel about this,im not sure how i feel about this
3,The service was excellent,the service was excellent
4,I had a bad experience,i had a bad experience


In [18]:

# Encode sentiment labels

encoder = LabelEncoder()

df['sentiment'] = encoder.fit_transform(df['sentiment'])

# Show encoded labels
df.head()

,message,sentiment,clean_message
0,I really enjoyed the movie,2,i really enjoyed the movie
1,The food was terrible,0,the food was terrible
2,I'm not sure how I feel about this,1,im not sure how i feel about this
3,The service was excellent,2,the service was excellent
4,I had a bad experience,0,i had a bad experience


In [19]:
# Tokenization

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df['clean_message'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['clean_message'])

print(sequences[:5])

[[3, 47, 137, 1, 38], [1, 34, 5, 54], [2, 13, 20, 69, 3, 138, 21, 4], [1, 25, 5, 74], [3, 55, 6, 95, 161]]


In [20]:
# Padding sequences

X = pad_sequences(sequences, maxlen=20)

print(X[:5])

[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   3  47 137
    1  38]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   1  34
    5  54]
 [  0   0   0   0   0   0   0   0   0   0   0   0   2  13  20  69   3 138
   21   4]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   1  25
    5  74]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   3  55   6
   95 161]]


In [21]:
# Target labels

y = df['sentiment']

print(y[:5])

0    2
1    0
2    1
3    2
4    0
Name: sentiment, dtype: int64


In [22]:
# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (467, 20)
Testing data: (117, 20)


In [23]:
# Build LSTM Model

model = Sequential()

# Embedding Layer
model.add(Embedding(input_dim = 5000, output_dim=128, input_length=20))

# LSTM Layer
model.add(LSTM(128))

# Dense Output Layer
model.add(Dense(3, activation='softmax'))

# Compile Model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Show model summary
model.summary()

C:\Users\prasa\OneDrive\Desktop\Data_Scientist\PHASE - II\tfenv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Train model

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 224ms/step - accuracy: 0.4433 - loss: 1.0578 - val_accuracy: 0.4615 - val_loss: 1.0190
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - accuracy: 0.4946 - loss: 0.9756 - val_accuracy: 0.6410 - val_loss: 0.9230
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.6809 - loss: 0.7811 - val_accuracy: 0.6154 - val_loss: 0.7098
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 101ms/step - accuracy: 0.7837 - loss: 0.5721 - val_accuracy: 0.7094 - val_loss: 0.5919
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - accuracy: 0.8608 - loss: 0.4175 - val_accuracy: 0.8547 - val_loss: 0.5048
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 111ms/step - accuracy: 0.9293 - loss: 0.2464 - val_accuracy: 0.8632 - val_loss: 0.3919
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - accuracy: 0.9572 - loss: 0.1429 - val_accuracy: 0.8803 - val_loss: 0.3629
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 102ms/step - accuracy: 0.9786 - loss: 0.0988 - val_accuracy: 0

In [25]:
# Evaluate model

loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy * 100)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8547 - loss: 0.3884
Accuracy: 85.47008633613586


In [28]:
# Predict custom message

text = ["semma movie da"]

# Clean text
text_seq = tokenizer.texts_to_sequences(text)

# Padding
text_pad = pad_sequences(text_seq, maxlen=20)

# Prediction
prediction = model.predict(text_pad)

# Get highest probability
predicted_class = prediction.argmax()

# Decode sentiment
labels = ['negative', 'neutral', 'positive']

print("Predicted Sentiment:", labels[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Predicted Sentiment: neutral


In [1]:
%%writefile requirements.txt
streamlit
scikit-learn
numpy
joblib
tensorflow==2.12.0


Overwriting requirements.txt


In [29]:
import pickle

# ✅ Save model (ONLY this way)
model.save("model.h5")

# ✅ Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

# ✅ Save encoder
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Files saved successfully")

✅ Files saved successfully
